## Regression Analysis

### Declaration
This analysis is **descriptive**. We estimate the conditional correlation 
between average monthly petrol prices and public transport patronage in 
Melbourne between 2022 and 2025, controlling for rainfall, population, public holidays, and seasonal patterns . We do not claim causality as petrol prices are likely correlated with other macroeconomic factors such as the post COVID recovery in commuting, these factors affected patronage at the same time period and cannot be fully detangled with the available data. Therefore, we report associations only and interpret results as correlations rather than causal effects.

### Econometric Specification

We estimate a log-linear OLS regression, where the dependant varibale is the natural log of total monthly public transport patronage. The log transofrrmation we thought was appropriate as patronage as seen int he EDA is right skewed and also mainly because it spans a large range of numbers (roughly 17-45 million trips a month). This gives the coefficient on petrol price a semi-elasticity interpretation- the percentage change in patronage associated with a one unit change in petrol price.

Our specification is;

ln(Patronage_t)  = β0 + β1·PetrolPrice_t + β2·Rainfall_t + β3·Population_t 
              + β4·PublicHolidays_t + Σγm·MonthDummy_m + ε_t

Where:

- Patronage_t = total monthly public transport patronage (persons)
- PetrolPrice_t = is our variable of interest. average monthly petrol price ($/L).  We expect β1 > 0 if higher driving costs shift commuters toward public transport.
- Rainfall_t = monthly rainfall (mm) - controls for weather effects on travel
- Population_t = monthly Melbourne population - controls for long-run growth trend unrelated to petrol prices.
- PublicHolidays_t = number of public holidays in the month - controls for 
reduced commuting days
- Month_dummies = eleven binary indicators for month of the year (January is the omitted baseline). These absorb seasonal patterns identified in the EDA, ensuring β1 captures only within-season variation in petrol prices rather than confounding seasonal trends. Per the Frisch-Waugh theorem, including these controls means β1 is identified from the variation in petrol prices orthogonal to all controls
- ε_t = error term

The unit of observation is a calendar month. The sample runs from 
January 2022 to September 2025 (N = 45 months) as this waa when reliable post-COVID data became available. OLS is appropriate given the continuous dependent variable and small monthly time series. Standard errors are reported as heteroskedasticity-robust (HC3) given the potential for non-constant variance in a time series context. This ensures valid inference regardless of the error structure as learned in the lecture material.

In [4]:
# Run regressions
model1 = sm.OLS(y, sm.add_constant(df[['PetrolPrice']])).fit(cov_type='HC3')
model2 = sm.OLS(y, X_base).fit(cov_type='HC3')
model3 = sm.OLS(y, X_full).fit(cov_type='HC3')

# Log-linear model for semi-elasticity interpretation
if 'log_patronage' not in df.columns:
    df['log_patronage'] = np.log(df['total_patronage'])
model4 = sm.OLS(df['log_patronage'], X_full).fit(cov_type='HC3')

print("Models estimated successfully")

Models estimated successfully


In [5]:
# Build a clean regression table
def make_table(models, model_names, dep_var, var_labels):
    rows = []
    for var, label in var_labels.items():
        coefs = []
        for m, model_name in zip(models, model_names):
            if var in m.params:
                coef = m.params[var]
                se = m.bse[var]
                pval = m.pvalues[var]
                stars = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
                
                # Use more decimals for the log-linear model
                if model_name == '(4) Log-Linear':
                    coef_str = f"{coef:.6f}{stars}"
                    se_str = f"{se:.6f}"
                elif abs(coef) < 100:
                    coef_str = f"{coef:.3f}{stars}"
                    se_str = f"{se:.3f}"
                else:
                    coef_str = f"{coef:,.0f}{stars}"
                    se_str = f"{se:,.2f}"
                
                coefs.append(f"{coef_str} ({se_str})")
            else:
                coefs.append("—")
        rows.append([label] + coefs)
    
    # Add fit stats
    rows.append(['N'] + [str(int(m.nobs)) for m in models])
    rows.append(['R²'] + [f"{m.rsquared:.3f}" for m in models])
    
    df_table = pd.DataFrame(rows, columns=['Variable'] + model_names)
    return df_table

var_labels = {
    'PetrolPrice': 'Petrol Price ($/L)',
    'rainfall_mm': 'Rainfall (mm)',
    'population': 'Population',
    'public_holiday_count': 'Public Holiday Count',
    'const': 'Constant'
}

from IPython.display import display

table = make_table(
    [model1, model2, model3, model4],
    ['(1) Bivariate', '(2) Controls', '(3) Controls + Season FE', '(4) Log-Linear'],
    'Total Patronage',
    var_labels
)

display(table)

,Variable,(1) Bivariate,(2) Controls,(3) Controls + Season FE,(4) Log-Linear
0,Petrol Price ($/L),"21,687,388 (16,126,313.14)","17,119,820 (11,190,860.86)","17,667,943** (7,448,140.31)",0.648278** (0.324025)
1,Rainfall (mm),—,"-5,794 (18,414.33)","-1,964 (23,060.25)",0.000109 (0.000774)
2,Population,—,26.379*** (5.721),27.094*** (5.069),0.000001*** (0.000000)
3,Public Holiday Count,—,"-532,963 (474,792.82)","-217,499 (849,933.36)",-0.004827 (0.033001)
4,Constant,"-2,553,521 (30,695,293.60)","-175,625,701*** (50,563,773.33)","-187,339,772*** (44,821,648.38)",10.562800*** (1.747144)
5,N,48,48,48,48
6,R²,0.141,0.543,0.829,0.791


### Regression Table Interpretation

The table above represents four OLS specifications, each one progressively adding more controls. reading across the columns allows us to see how the coefficient on petrol price fluctuates as we add in more possible confounders (useful diagnostic for OVB). If the coefficient changes subtantially between models, it suggests the simpler models were pickinng up confounding variation, or, if it remains stable it suggests the association is resilient to the inclusion of additional controls.

The petrol price coefficient remains positive and broadly stable across all four models, which explains that petrol patronage is not just an artefact of omitted controls. As we see the R^2 rises substantially from 0.141 in model 1 to 0.823 in model 3, this reflects the large explanatory power of seasonal patterns and population growth once added (explains 82% of variation in public transport usage). Our preferred specification is model 4, the log-linear model...

Although Model 4 has a slightly lower R^2 (0.791) than Model 3 (0.829), this doesn't mean Model (4) is a worse specification. The R^2 values are not directly comparable across the two models because they have different dependent variables- Model 3 explains variation in patronage levels while Model 4 explains variation in log patronage. A lower R^2 reflects that the model is explaining a different transformation of the outcome variable, not that it fits the data worse.
Therefore, Model 4 is preferred for a few reasons, firstly as discussed in the declaration is addressed the righyt-skewed distribution of patronage. Secondly, the semi-elasticity interpretation is more economically meaningful than a trip count. And, percenage changes are comparable across different baseline levels of patronage, ths makes the coefficient more interpretable regardless of whether patronage is at 20 million or 40 million.


### Interpretation of Main Coefficient

The coefficient of interest is β1 on petrol price in model 4, our preferred log-linear specification.

**Direction:** The coefficient is positive (β1 = 0.648), indicating that an increase in petrol prices is associated with higher public transport patronage. This is consistent with economic intuition, as the cost of driving increases, some commuters substitute toward public transport.

**Magnitude:** A $1.00 per litre increase in the average monthly petrol price is associated with approximately a 64.8% increase in monthly public transport patronage. Or a more realistic $0.50 per litre increase- consistent with price movements observed within the sample period - is associated with approximately a 32.4% increase in patronage.

**Units:** Petrol price is measured in dollars per litre- $/L. Because the dependant variable is log transformed the coefficient is interpreted as a semi-elasticity... a one unit increase petrol price is associated with 
an approximate β1 × 100% change in patronage.

**What is held constant:** This association is estimated holding monthly rainfall, Melbourne population, public holiday count, and seasonal patterns (month of the year fixed effects) constant. Per the Frisch-Waugh theorem, β1 
is identified from the variation in petrol prices that cannot be explained by any of these controls.

**Supporting coefficients:** Population is positive and statistically significant (β = 0.000001, p < 0.01), consistent with long-run growth in transit demand as Melbourne grows. Rainfall and public holiday count are 
both statistically insignificant in the preferred specification, suggesting limited independent variation once seasonality is controlled for.

### Threats and Limitations

Since this analysis is descriptive, we do not claim to identify a causal effect. We report conditional correlations only. The main limitations are:

**1. Omitted Variable Bias**
The most significant threat is the post-COVID recovery. In early 2022, petrol prices spiked due to global supply shocks at the same time that Melbourne commuters were returning to work as COVID restrictions lifted. Both forces increased patronage simultaneously, meaning our model cannot fully disentangle the petrol price effect from the return to work effect. 

Using the OVB formula:

OVB = β2 × γ1

Where β2 (effect of COVID recovery on patronage) is positive and γ1 (correlation between COVID recovery and petrol price) is positive, giving a positive OVB. This means our coefficient of 0.648 is likely 
upward biased (most likely inflting the result) so the true association between petrol prices and patronage is probably smaller than we estimate. We partially address this by including month fixed effects and population as controls, but the COVID recovery effect cannot be fully removed without a longer pre-COVID baseline.

**2. Exogeneity**
Exogeneity requires E[ε | PetrolPrice] = 0  that knowing the petrol price tells us nothing about unobserved factors affecting patronage in the error term. This is unlikely to hold in our case as unobserved factors such as broader economic conditions, consumer confidence, and employment levels all sit in the error term and are correlated with petrol prices. This is precisely why we declare our analysis descriptive rather than causal- we cannot credibly satisfy the exogeneity assumption with the available data.

**3. Possible Reverse Causality**
Petrol prices are assumed to be exogenous to Melbourne public transport patronage in this projet, so that is, patronage does not affect petrol prices. This is plausible at the city level since Melbourne's transport demand is too small to move global or national fuel prices. However, broader economic activity in Victoria could simultaneously drive both higher commuting demand and higher fuel consumption, creating a spurious positive 
association. Therefore, this is a limitation thsat is unlikely, but we cannot fully rule it out.

**4. Degrees of Freedom**
With only 48 monthly observations and 15 parameters (4 controls + 11 month dummies + constant), the model has 32 degrees of freedom remaining. This limits statistical power and means results may be sensitive to individual influential months- particularly the low patronage COVID affected months in early 2022. A longer time series would therefore considerably strengthen the reliability of these estimates and conclusions.

**5. Measurement Error**
Petrol prices, population growth and transport patronage are all measured at the Victorian state level, this isn't strictly the melbourne commuter market. If regional travel behaviour differs from metropolitan patterns- for example, rural areas having fewer public transport substitutes for driving- this could bias the estimated association in either direction.However, in our EDA we examined the distribution of metropolitan and regional patronage separately, finding that metropolitan patronage variables are concentrated in the higher range while regional variables such as regional train and coach show more uneven and irregular distributions. This suggests metropolitan Melbourne dominates the overall Victorian patronage series, partially mitigating this concern. In addition, population is interpolated from quarterly ABS data into monthly values, introducing some measurement imprecision.

**6. Sample Selection**
The sample begins in January 2022 during the post-COVID recovery period. This means the dataset excludes the pre-COVID period where the relationship between petrol prices and patronage may have looked 
very different. The results should therefore be interpreted as describing the post-COVID period specifically, and may not generalise to other time periods.


In [19]:
import os
os.makedirs('output', exist_ok=True)
table.to_csv('output/regression_table.csv', index=False)
print("Regression table saved to output/regression_table.csv")

Regression table saved to output/regression_table.csv
